<a href="https://colab.research.google.com/github/Arham-15/traffic-prediction-system/blob/dev/notebooks/03_feature_engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Feature Engineering

This notebook prepares the cleaned traffic dataset for machine learning by creating useful time-based and weather-related features.


In [2]:
import pandas as pd

df = pd.read_csv(r"/content/traffic_cleaned.csv")

df["date_time"] = pd.to_datetime(df["date_time"])

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

Shape: (48187, 9)

Columns:
['holiday', 'temp', 'rain_1h', 'snow_1h', 'clouds_all', 'weather_main', 'weather_description', 'date_time', 'traffic_volume']


In [3]:
df["year"] = df["date_time"].dt.year
df["month"] = df["date_time"].dt.month
df["day"] = df["date_time"].dt.day
df["hour"] = df["date_time"].dt.hour
df["day_of_week"] = df["date_time"].dt.dayofweek
df["is_weekend"] = (df["day_of_week"] >= 5).astype(int)

df.head()

,holiday,temp,rain_1h,snow_1h,clouds_all,weather_main,weather_description,date_time,traffic_volume,year,month,day,hour,day_of_week,is_weekend
0,NaN,288.28,0.0,0.0,40,Clouds,scattered clouds,2012-10-02 09:00:00,5545,2012,10,2,9,1,0
1,NaN,289.36,0.0,0.0,75,Clouds,broken clouds,2012-10-02 10:00:00,4516,2012,10,2,10,1,0
2,NaN,289.58,0.0,0.0,90,Clouds,overcast clouds,2012-10-02 11:00:00,4767,2012,10,2,11,1,0
3,NaN,290.13,0.0,0.0,90,Clouds,overcast clouds,2012-10-02 12:00:00,5026,2012,10,2,12,1,0
4,NaN,291.14,0.0,0.0,75,Clouds,broken clouds,2012-10-02 13:00:00,4918,2012,10,2,13,1,0


In [4]:
print(df.shape)
print(df.columns.tolist())

(48187, 15)
['holiday', 'temp', 'rain_1h', 'snow_1h', 'clouds_all', 'weather_main', 'weather_description', 'date_time', 'traffic_volume', 'year', 'month', 'day', 'hour', 'day_of_week', 'is_weekend']


In [5]:
print("Weather Main categories:")
print(df["weather_main"].value_counts())

print("\nWeather Description categories:")
print(df["weather_description"].value_counts())

Weather Main categories:
weather_main
Clouds          15158
Clear           13384
Mist             5949
Rain             5672
Snow             2875
Drizzle          1820
Haze             1360
Thunderstorm     1033
Fog               912
Smoke              20
Squall              4
Name: count, dtype: int64

Weather Description categories:
weather_description
sky is clear                           11659
mist                                    5949
overcast clouds                         5079
broken clouds                           4663
scattered clouds                        3460
light rain                              3372
few clouds                              1956
light snow                              1945
Sky is Clear                            1725
moderate rain                           1664
haze                                    1360
light intensity drizzle                 1099
fog                                      912
proximity thunderstorm                   673
drizzle    

In [12]:
df = df.drop(columns=["weather_description"], errors="ignore")

print("Shape after removing weather_description:", df.shape)
print(df.columns.tolist())

Shape after removing weather_description: (48187, 14)
['holiday', 'temp', 'rain_1h', 'snow_1h', 'clouds_all', 'weather_main', 'date_time', 'traffic_volume', 'year', 'month', 'day', 'hour', 'day_of_week', 'is_weekend']


In [13]:
print("Holiday values:")
print(df["holiday"].value_counts(dropna=False))

print("\nUnique holiday values:")
print(df["holiday"].unique())

Holiday values:
holiday
NaN                          48126
Labor Day                        7
Christmas Day                    6
Thanksgiving Day                 6
Martin Luther King Jr Day        6
New Years Day                    6
Veterans Day                     5
Columbus Day                     5
Memorial Day                     5
Washingtons Birthday             5
State Fair                       5
Independence Day                 5
Name: count, dtype: int64

Unique holiday values:
[nan 'Columbus Day' 'Veterans Day' 'Thanksgiving Day' 'Christmas Day'
 'New Years Day' 'Washingtons Birthday' 'Memorial Day' 'Independence Day'
 'State Fair' 'Labor Day' 'Martin Luther King Jr Day']


In [19]:
if "holiday" in df.columns:
    # Create binary holiday feature
    df["is_holiday"] = df["holiday"].notna().astype(int)
    # Remove original holiday column
    df = df.drop(columns=["holiday"])
    print(df["is_holiday"].value_counts())
elif "is_holiday" in df.columns:
    print("Column 'holiday' was already processed. 'is_holiday' column exists:")
    print(df["is_holiday"].value_counts())
else:
    print("Neither 'holiday' nor 'is_holiday' columns found. Cannot create 'is_holiday'.")

Column 'holiday' was already processed. 'is_holiday' column exists:
is_holiday
0    48126
1       61
Name: count, dtype: int64


In [20]:
df = pd.get_dummies(
    df,
    columns=["weather_main"],
    prefix="weather",
    dtype=int
)

print("Shape:", df.shape)
print(df.columns.tolist())

Shape: (48187, 24)
['temp', 'rain_1h', 'snow_1h', 'clouds_all', 'date_time', 'traffic_volume', 'year', 'month', 'day', 'hour', 'day_of_week', 'is_weekend', 'is_holiday', 'weather_Clear', 'weather_Clouds', 'weather_Drizzle', 'weather_Fog', 'weather_Haze', 'weather_Mist', 'weather_Rain', 'weather_Smoke', 'weather_Snow', 'weather_Squall', 'weather_Thunderstorm']


In [22]:
print("Dataset shape:", df.shape)

print("\nMissing values:")
print(df.isnull().sum()[df.isnull().sum() > 0])

print("\nDuplicate rows:", df.duplicated().sum())

print("\nData types:")
print(df.dtypes)

Dataset shape: (48187, 24)

Missing values:
Series([], dtype: int64)

Duplicate rows: 217

Data types:
temp                           float64
rain_1h                        float64
snow_1h                        float64
clouds_all                       int64
date_time               datetime64[ns]
traffic_volume                   int64
year                             int32
month                            int32
day                              int32
hour                             int32
day_of_week                      int32
is_weekend                       int64
is_holiday                       int64
weather_Clear                    int64
weather_Clouds                   int64
weather_Drizzle                  int64
weather_Fog                      int64
weather_Haze                     int64
weather_Mist                     int64
weather_Rain                     int64
weather_Smoke                    int64
weather_Snow                     int64
weather_Squall                   int64


In [23]:
duplicates = df[df.duplicated(keep=False)].sort_values("date_time")

print("Number of rows involved in exact duplicates:", len(duplicates))

duplicates.head(20)

Number of rows involved in exact duplicates: 425


,temp,rain_1h,snow_1h,clouds_all,date_time,traffic_volume,year,month,day,hour,...,weather_Clouds,weather_Drizzle,weather_Fog,weather_Haze,weather_Mist,weather_Rain,weather_Smoke,weather_Snow,weather_Squall,weather_Thunderstorm
358,285.72,0.0,0.0,100,2012-10-17 09:00:00,5403,2012,10,17,9,...,0,0,0,0,0,1,0,0,0,0
359,285.72,0.0,0.0,100,2012-10-17 09:00:00,5403,2012,10,17,9,...,0,0,0,0,0,1,0,0,0,0
403,279.79,0.0,0.0,87,2012-10-19 04:00:00,720,2012,10,19,4,...,0,1,0,0,0,0,0,0,0,0
406,279.79,0.0,0.0,87,2012-10-19 04:00:00,720,2012,10,19,4,...,0,1,0,0,0,0,0,0,0,0
432,280.43,0.0,0.0,92,2012-10-19 22:00:00,3077,2012,10,19,22,...,0,1,0,0,0,0,0,0,0,0
433,280.43,0.0,0.0,92,2012-10-19 22:00:00,3077,2012,10,19,22,...,0,1,0,0,0,0,0,0,0,0
437,280.06,0.0,0.0,92,2012-10-20 00:00:00,1270,2012,10,20,0,...,0,0,0,0,0,1,0,0,0,0
438,280.06,0.0,0.0,92,2012-10-20 00:00:00,1270,2012,10,20,0,...,0,0,0,0,0,1,0,0,0,0
460,281.70,0.0,0.0,100,2012-10-20 09:00:00,3713,2012,10,20,9,...,0,1,0,0,0,0,0,0,0,0
462,281.70,0.0,0.0,100,2012-10-20 09:00:00,3713,2012,10,20,9,...,0,1,0,0,0,0,0,0,0,0


In [24]:
duplicate_counts = (
    df.value_counts()
      .reset_index(name="count")
)

duplicate_counts[duplicate_counts["count"] > 1].head(20)

,temp,rain_1h,snow_1h,clouds_all,date_time,traffic_volume,year,month,day,hour,...,weather_Drizzle,weather_Fog,weather_Haze,weather_Mist,weather_Rain,weather_Smoke,weather_Snow,weather_Squall,weather_Thunderstorm,count
0,290.88,0.0,0.0,90,2013-05-31 02:00:00,339,2013,5,31,2,...,0,0,0,0,1,0,0,0,0,3
1,291.39,0.0,0.0,90,2013-05-31 00:00:00,741,2013,5,31,0,...,0,0,0,0,0,0,0,0,1,3
2,287.15,0.0,0.0,90,2013-05-19 10:00:00,3591,2013,5,19,10,...,0,0,0,0,0,0,0,0,1,3
3,276.92,0.0,0.0,90,2013-03-31 07:00:00,908,2013,3,31,7,...,0,0,0,0,1,0,0,0,0,3
4,290.99,0.0,0.0,90,2013-06-01 02:00:00,557,2013,6,1,2,...,0,0,0,0,0,0,0,0,1,3
5,282.71,0.0,0.0,100,2012-10-20 12:00:00,4613,2012,10,20,12,...,0,0,0,0,1,0,0,0,0,3
6,286.36,0.0,0.0,90,2013-05-19 09:00:00,2802,2013,5,19,9,...,0,0,0,0,0,0,0,0,1,3
7,286.78,0.0,0.0,90,2013-05-19 06:00:00,815,2013,5,19,6,...,0,0,0,0,0,0,0,0,1,3
8,292.08,0.0,0.0,90,2013-05-20 07:00:00,6136,2013,5,20,7,...,0,0,0,0,0,0,0,0,1,3
9,278.08,0.0,0.0,90,2013-03-31 05:00:00,359,2013,3,31,5,...,0,0,0,0,1,0,0,0,0,2


In [26]:
before = len(df)

df = df.drop_duplicates().reset_index(drop=True)

after = len(df)

print("Rows before:", before)
print("Rows after:", after)
print("Duplicate rows removed:", before - after)
print("Remaining duplicate rows:", df.duplicated().sum())

Rows before: 47970
Rows after: 47970
Duplicate rows removed: 0
Remaining duplicate rows: 0


In [28]:
timestamp_counts = df["date_time"].value_counts()

print("Unique timestamps:", timestamp_counts.size)
print("Timestamps appearing more than once:",
      (timestamp_counts > 1).sum())

print("\nMaximum rows for a single timestamp:",
      timestamp_counts.max())

Unique timestamps: 40575
Timestamps appearing more than once: 5402

Maximum rows for a single timestamp: 5


In [29]:
duplicate_timestamps = (
    df[df["date_time"].duplicated(keep=False)]
    .sort_values("date_time")
)

duplicate_timestamps[
    ["date_time", "traffic_volume", "temp", "rain_1h",
     "snow_1h", "clouds_all"]
].head(20)

,date_time,traffic_volume,temp,rain_1h,snow_1h,clouds_all
178,2012-10-10 07:00:00,6793,281.25,0.0,0.0,99
179,2012-10-10 07:00:00,6793,281.25,0.0,0.0,99
180,2012-10-10 08:00:00,6283,280.10,0.0,0.0,99
181,2012-10-10 08:00:00,6283,280.10,0.0,0.0,99
182,2012-10-10 09:00:00,5680,279.61,0.0,0.0,99
183,2012-10-10 09:00:00,5680,279.61,0.0,0.0,99
269,2012-10-14 09:00:00,2685,282.43,0.0,0.0,57
270,2012-10-14 09:00:00,2685,282.43,0.0,0.0,57
271,2012-10-14 09:00:00,2685,282.43,0.0,0.0,57
272,2012-10-14 10:00:00,3370,282.33,0.0,0.0,57


In [30]:
# Investigate what differs between rows sharing the same timestamp

timestamp_analysis = df.groupby("date_time").agg(
    rows=("date_time", "size"),
    unique_traffic=("traffic_volume", "nunique"),
    unique_temp=("temp", "nunique"),
    unique_rain=("rain_1h", "nunique"),
    unique_snow=("snow_1h", "nunique"),
    unique_clouds=("clouds_all", "nunique")
)

# find timestamps where multiple observations have different values
timestamp_analysis[
    (timestamp_analysis["rows"] > 1) &
    (
        (timestamp_analysis["unique_traffic"] > 1) |
        (timestamp_analysis["unique_temp"] > 1) |
        (timestamp_analysis["unique_rain"] > 1) |
        (timestamp_analysis["unique_snow"] > 1) |
        (timestamp_analysis["unique_clouds"] > 1)
    )
].head(20)

,rows,unique_traffic,unique_temp,unique_rain,unique_snow,unique_clouds
date_time,,,,,,
2012-11-04 01:00:00,2,1,2,1,1,1
2014-01-03 20:00:00,2,1,2,1,1,1
2014-01-09 16:00:00,2,1,2,1,1,2
2014-01-11 16:00:00,2,1,2,1,1,2
2014-01-15 00:00:00,2,1,2,1,1,2
2014-01-15 16:00:00,2,1,2,1,1,2
2014-01-16 16:00:00,2,1,2,1,1,2
2014-01-17 03:00:00,2,1,2,1,1,2
2014-01-17 06:00:00,2,1,2,1,1,2


In [31]:
print("Duplicate timestamps with different traffic values:",
      ((timestamp_analysis["rows"] > 1) &
       (timestamp_analysis["unique_traffic"] > 1)).sum())

print("Duplicate timestamps with different temperature values:",
      ((timestamp_analysis["rows"] > 1) &
       (timestamp_analysis["unique_temp"] > 1)).sum())

print("Duplicate timestamps with different rain values:",
      ((timestamp_analysis["rows"] > 1) &
       (timestamp_analysis["unique_rain"] > 1)).sum())

print("Duplicate timestamps with different snow values:",
      ((timestamp_analysis["rows"] > 1) &
       (timestamp_analysis["unique_snow"] > 1)).sum())

print("Duplicate timestamps with different cloud values:",
      ((timestamp_analysis["rows"] > 1) &
       (timestamp_analysis["unique_clouds"] > 1)).sum())

Duplicate timestamps with different traffic values: 0
Duplicate timestamps with different temperature values: 78
Duplicate timestamps with different rain values: 9
Duplicate timestamps with different snow values: 0
Duplicate timestamps with different cloud values: 32


In [32]:
weather_columns = [
    "weather_Clear",
    "weather_Clouds",
    "weather_Drizzle",
    "weather_Fog",
    "weather_Haze",
    "weather_Mist",
    "weather_Rain",
    "weather_Smoke",
    "weather_Snow",
    "weather_Squall",
    "weather_Thunderstorm"
]

weather_per_timestamp = df.groupby("date_time")[weather_columns].sum()

conflicting_weather = weather_per_timestamp[
    weather_per_timestamp.sum(axis=1) > 1
]

print("Timestamps with multiple weather categories:",
      len(conflicting_weather))

conflicting_weather.head(20)

Timestamps with multiple weather categories: 5402


,weather_Clear,weather_Clouds,weather_Drizzle,weather_Fog,weather_Haze,weather_Mist,weather_Rain,weather_Smoke,weather_Snow,weather_Squall,weather_Thunderstorm
date_time,,,,,,,,,,,
2012-10-10 07:00:00,0,0,1,0,0,0,1,0,0,0,0
2012-10-10 08:00:00,0,0,1,0,0,0,1,0,0,0,0
2012-10-10 09:00:00,0,0,1,0,0,0,1,0,0,0,0
2012-10-14 09:00:00,0,0,1,0,1,1,0,0,0,0,0
2012-10-14 10:00:00,0,0,1,0,0,1,0,0,0,0,0
2012-10-14 11:00:00,0,0,1,0,0,1,0,0,0,0,0
2012-10-14 12:00:00,0,0,1,0,0,1,1,0,0,0,0
2012-10-14 13:00:00,0,0,1,0,0,1,1,0,0,0,0
2012-10-14 14:00:00,0,0,1,0,0,1,1,0,0,0,0


In [33]:
weather_features = [
    "temp",
    "rain_1h",
    "snow_1h",
    "clouds_all"
]

weather_encoded = [
    "weather_Clear",
    "weather_Clouds",
    "weather_Drizzle",
    "weather_Fog",
    "weather_Haze",
    "weather_Mist",
    "weather_Rain",
    "weather_Smoke",
    "weather_Snow",
    "weather_Squall",
    "weather_Thunderstorm"
]

time_features = [
    "year",
    "month",
    "day",
    "hour",
    "day_of_week",
    "is_weekend",
    "is_holiday"
]

aggregation_rules = {
    "traffic_volume": "first",
    "temp": "mean",
    "rain_1h": "mean",
    "snow_1h": "mean",
    "clouds_all": "mean",
    "year": "first",
    "month": "first",
    "day": "first",
    "hour": "first",
    "day_of_week": "first",
    "is_weekend": "first",
    "is_holiday": "first"
}

# Average one-hot weather features
for col in weather_encoded:
    aggregation_rules[col] = "mean"

df_aggregated = (
    df.groupby("date_time", as_index=False)
      .agg(aggregation_rules)
)

print("Original rows:", len(df))
print("Aggregated rows:", len(df_aggregated))
print("Unique timestamps:", df["date_time"].nunique())

Original rows: 47970
Aggregated rows: 40575
Unique timestamps: 40575


In [34]:
print("Duplicate timestamps:", df_aggregated["date_time"].duplicated().sum())
print("Missing values:", df_aggregated.isnull().sum().sum())
print("Shape:", df_aggregated.shape)

Duplicate timestamps: 0
Missing values: 0
Shape: (40575, 24)


In [36]:
df_aggregated.head()

,date_time,traffic_volume,temp,rain_1h,snow_1h,clouds_all,year,month,day,hour,...,weather_Clouds,weather_Drizzle,weather_Fog,weather_Haze,weather_Mist,weather_Rain,weather_Smoke,weather_Snow,weather_Squall,weather_Thunderstorm
0,2012-10-02 09:00:00,5545,288.28,0.0,0.0,40.0,2012,10,2,9,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2012-10-02 10:00:00,4516,289.36,0.0,0.0,75.0,2012,10,2,10,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2012-10-02 11:00:00,4767,289.58,0.0,0.0,90.0,2012,10,2,11,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,2012-10-02 12:00:00,5026,290.13,0.0,0.0,90.0,2012,10,2,12,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,2012-10-02 13:00:00,4918,291.14,0.0,0.0,75.0,2012,10,2,13,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [37]:
df = df_aggregated.copy()

df = df.sort_values("date_time").reset_index(drop=True)

print("Shape:", df.shape)
print("Start:", df["date_time"].min())
print("End:", df["date_time"].max())

Shape: (40575, 24)
Start: 2012-10-02 09:00:00
End: 2018-09-30 23:00:00


In [38]:
features = [col for col in df.columns
            if col not in ["date_time", "traffic_volume"]]

X = df[features]
y = df["traffic_volume"]

print("Number of features:", len(features))
print("\nFeatures:")
print(features)

print("\nTarget:", y.name)

Number of features: 22

Features:
['temp', 'rain_1h', 'snow_1h', 'clouds_all', 'year', 'month', 'day', 'hour', 'day_of_week', 'is_weekend', 'is_holiday', 'weather_Clear', 'weather_Clouds', 'weather_Drizzle', 'weather_Fog', 'weather_Haze', 'weather_Mist', 'weather_Rain', 'weather_Smoke', 'weather_Snow', 'weather_Squall', 'weather_Thunderstorm']

Target: traffic_volume


In [42]:
df.to_csv("traffic_features.csv", index=False)

print("Feature-engineered dataset saved.")

Feature-engineered dataset saved.
